# VESTIGIUM | NASA FIRMS Indonesia EDA (2000–2025)
**NASA Space Apps Challenge 2026 | Team Ga Tidur**

This reproducible exploratory analysis covers **48 Indonesia CSV files in `data.zip`**. The raw-source audit was performed on the **entire dataset**, not a sample; an unrelated Angola file was excluded. Aggregated tables and all 17 charts are bundled so this notebook can be opened without reprocessing millions of rows.

**Unit of observation:** one record is **one thermal-anomaly detection at the satellite-pixel level** during an overpass. It is not a distinct fire incident or an estimate of burned area. `type=0` indicates *presumed vegetation fire*, **not** evidence of deliberate burning.


## 0. Setup and reproducibility

Open this notebook from the directory containing `tables/`, `figures/`, and `summary.json`. The original aggregates were generated from `data.zip` by `vestigium_eda.py` using chunked CSV reads.

To **rebuild the original data outputs**, place `data.zip` in this directory and run `python vestigium_eda.py data.zip .` followed by `python render_eda.py`. The notebook does not automatically run this expensive raw-data processing step.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
T = ROOT / 'tables'
F = ROOT / 'figures'
assert (ROOT/'summary.json').is_file() and T.is_dir(), 'Buka notebook dari folder VESTIGIUM_EDA (dengan tables/ dan summary.json).'
summary = json.loads((ROOT/'summary.json').read_text(encoding='utf-8'))
print('Working directory:', ROOT)
print('Source archive:', summary['data_zip'])
print('Indonesia records:', f"{summary['all_indonesia_records']:,}")
print('Total type=0 records:', f"{summary['type0_total']:,}")

## 1. File inventory and observation coverage

Review the analyzed files, years, row counts, and first/last observations. A year in a filename does not guarantee a full calendar year of observations.


In [ ]:
inventory = pd.read_csv(T/'file_inventory.csv')
display(inventory.groupby('sensor').agg(files=('filename','count'), total=('rows_all','sum'), presumed_vegetation=('rows_type0','sum'), first=('first_date_utc','min'), last=('last_date_utc','max')).reset_index())
print('Excluded files:', summary['excluded_files'])
display(inventory.head(10))

## 2. Schema and data dictionary

MODIS provides `brightness`/`bright_t31`, whereas VIIRS provides `bright_ti4`/`bright_ti5`. These sensor-specific brightness measurements are **not directly interchangeable**.


In [ ]:
for source, columns in summary['schemas'].items():
    print(f'{source}: {len(columns)} columns')
    print(', '.join(columns))
print('Within-sensor schema changes:', len(summary['schema_drift']))

**Key fields:** `latitude` and `longitude` are pixel centroids; `acq_date` and `acq_time` are **UTC** acquisition values; `type` indicates thermal-anomaly category; `frp` is Fire Radiative Power in MW; `confidence` is a sensor-specific confidence indicator; and `version` identifies the source product/processing label.

For a defensible API, preserve sensor, type and product-version metadata **on every record**.


## 3. Column-level missingness audit

A displayed zero means **no missing values were parsed in that CSV column** under the implemented checks. It does not prove that each physical measurement is accurate.


In [ ]:
missing = pd.read_csv(T/'missingness_by_source_column.csv')
print('Total missing values:', int(missing.null_count.sum()))
display(missing.pivot(index='column',columns='source',values='null_pct').fillna(0).round(2))

## 4. Domain, temporal, spatial, and duplicate validation

Checks cover UTC date/time formatting, file-year consistency, global coordinate bounds, an **approximate Indonesia bounding box**, FRP and type values. The bounding box is not an authoritative Indonesia land-polygon test. Full-row hashes check **within-file** duplicates, not repeated physical detections across sensors.


In [ ]:
validity = pd.read_csv(T/'validity_checks.csv')
display(validity.pivot(index='check',columns='source',values='count').fillna(0).astype(int))
print('Exact within-file duplicate rows:', inventory.duplicate_full_rows_within_file.sum())
assert summary['all_indonesia_records'] == inventory.rows_all.sum()
assert summary['type0_total'] == inventory.rows_type0.sum()
assert inventory.duplicate_full_rows_within_file.sum() == 0
print('Aggregate consistency check: PASS')

## 5. Thermal-anomaly type composition

`type=0`: presumed vegetation fire; `1`: volcanic; `2`: static heat source; `3`: offshore. The primary hotspot exploration uses `type=0`, while other classes remain relevant to data-quality auditing.


In [ ]:
types = pd.read_csv(T/'type_distribution.csv')
display(types.pivot_table(index='source',columns='type',values='count',aggfunc='sum').fillna(0).astype(int))
display(Image(filename=str(F/'07_type_mix_by_sensor.png')))

## 6. Product versions and provenance

Differences in the raw `version` field may reflect different product-processing histories. Do not silently rewrite different labels to one common version without verified product metadata.


In [ ]:
versions = pd.read_csv(T/'versions.csv',dtype={'version':'string'})
display(versions)
display(Image(filename=str(F/'14_modis_versions.png')))

## 7. Annual detection counts by sensor

Interpret each series separately. VIIRS nominal 375 m detections may capture smaller fires than MODIS at approximately 1 km. Adding counts across instruments can count the same physical activity repeatedly.


In [ ]:
annual = pd.read_csv(T/'annual_type0.csv')
for source in ['MODIS','VIIRS S-NPP','VIIRS NOAA-20']:
    x = annual[annual.source==source].sort_values('type0_count',ascending=False)
    print('\n',source,'peak years:', x[['year','type0_count']].head(4).to_string(index=False))
for name in ['01_annual_MODIS.png','02_annual_VIIRS_S_NPP.png','03_annual_VIIRS_NOAA_20.png']:
    display(Image(filename=str(F/name)))

## 8. Monthly timeline and year-by-month heatmaps

Use UTC observation dates. Note incomplete starting years: MODIS 2000, VIIRS S-NPP 2012, and NOAA-20 2018. A gap with no detection is **not proof that no fires occurred**.


In [ ]:
monthly = pd.read_csv(T/'monthly_type0.csv')
display(monthly.nlargest(15,'type0_count')[['source','date_month','type0_count','frp_sum_mw']])
for name in ['04_calendar_MODIS.png','05_calendar_VIIRS_S-NPP.png']:
    display(Image(filename=str(F/name)))

## 9. Seasonality and within-year normalization

Compute monthly detection counts as a share of each **individual sensor's** annual total, then average those shares across years. This reduces raw-count scaling effects for seasonal comparisons but **does not harmonize MODIS and VIIRS detection characteristics**.


In [ ]:
for source,start in [('MODIS',2001),('VIIRS S-NPP',2013),('VIIRS NOAA-20',2019)]:
    sub = monthly[(monthly.source==source)&(monthly.year>=start)].copy()
    tot = sub.groupby('year').type0_count.transform('sum')
    sub['pct_year'] = 100*sub.type0_count/tot
    top_month = sub.groupby('month').pct_year.mean().nlargest(3)
    print(source, 'Highest average month shares (%):', top_month.round(2).to_dict())
display(Image(filename=str(F/'06_seasonality_share.png')))

## 10. Daytime versus nighttime

Day/night proportions depend partly on satellite overpass scheduling. They must not be used as fire ignition or extinction times.


In [ ]:
dn = pd.read_csv(T/'daynight_distribution.csv')
display(dn)
display(Image(filename=str(F/'08_day_night_mix.png')))

## 11. FRP distribution, outliers, and long right tails

FRP measures **pixel radiative power**, in MW. The following statistics include P25, median, P75, P95, P99, mean, and maximum from **all `type=0` observations**. With a heavy right tail, the median is generally more representative than the mean. MODIS and VIIRS FRP cannot be interpreted as identical measures of the physical intensity of an individual incident without further study.


In [ ]:
frp = pd.read_csv(T/'frp_type0_stats.csv')
cols = ['source','n_type0','frp_p25_mw','frp_p50_mw','frp_p75_mw','frp_p95_mw','frp_p99_mw','frp_mean_mw','frp_p100_mw','zero_frp_n']
display(frp[cols].round(2))
display(Image(filename=str(F/'09_frp_quantiles.png')))

## 12. Sensor-specific confidence scales

MODIS confidence 0–100 is an algorithmic indicator, not necessarily a calibrated probability. VIIRS uses low/nominal/high labels. **There is no automatic valid mapping** between these scales.


In [ ]:
confidence = pd.read_csv(T/'confidence_distribution.csv',dtype={'confidence':'string'})
display(confidence[confidence.source.str.contains('VIIRS')].sort_values(['source','count'],ascending=[True,False]).head(10))
print('MODIS type=0 confidence percentiles:', summary['confidence_notes']['modis_type0_confidence_quantiles'])
display(Image(filename=str(F/'10_viirs_confidence.png')))

## 13. Yearly spatial distribution on a 0.5° grid

These bins form a coarse geographic EDA grid. A 0.5° cell is unsuitable for precise province assignment and is not a burned-area measure. Official Indonesia boundaries have not yet been used for masking or coastlines.


In [ ]:
grid = pd.read_csv(T/'spatial_grid_0p5deg_type0_by_year.csv')
print('Grid/year/source aggregated rows:',len(grid))
display(grid[(grid.source=='VIIRS S-NPP')&(grid.year==2015)].nlargest(12,'count'))
for name in ['11_spatial_density_2015.png','12_spatial_density_2019.png']:
    display(Image(filename=str(F/name)))

## 14. Comparing sensors within the same year

The 2015 comparison uses MODIS and S-NPP; 2019 contains all three included sensor groups. Normalized comparisons characterize the **shape of monthly seasonality**, not corrected detection bias or the true number of fire incidents.


In [ ]:
display(annual[annual.year.isin([2015,2019])][['source','year','type0_count','frp_sum_mw']].sort_values(['year','source']))
for name in ['13_normalized_2015_sensor_comparison.png','15_raw_2019_comparison.png']:
    display(Image(filename=str(F/name)))

## 15. Descriptive 2019 grid-presence comparison

The Jaccard index below summarizes overlap between **occupied monthly grid cells**, each containing at least one detection. It does not match individual same-time detections and is not an ML accuracy score.


In [ ]:
overlap = pd.read_csv(T/'2019_monthly_spatial_grid_overlap.csv')
display(overlap.round(3))
print('2019 median Jaccard index:', round(overlap.jaccard_presence.median(),3))
display(Image(filename=str(F/'17_spatial_overlap_grid2019.png')))

## 16. Extreme dates and high-detection days

The twelve largest daily detection counts do not rank disasters by burned area or casualties. Fire-event analysis requires explicit spatiotemporal grouping and independent verification.


In [ ]:
daily = pd.read_csv(T/'daily_type0.csv')
display(daily[daily.source=='VIIRS S-NPP'].nlargest(12,'type0_count')[['date_utc','type0_count','frp_sum_mw']])
display(Image(filename=str(F/'16_peak_dates_snpp.png')))

## 17. Partial years, observation gaps, and aggregate validation

**Coverage caveat:** MODIS 2000 starts on 1 November; S-NPP 2012 on 20 January; NOAA-20 2018 on 1 April. Missing dates can indicate no reported detection or observation limitations. Do not force absent dates to represent zero activity in ML without specifying observation coverage or exposure.


In [ ]:
coverage = pd.DataFrame([{'source':s,'first_utc':d['first'],'last_utc':d['last'],'days_with_records':d['days_with_at_least_one_detection_all_types']} for s,d in summary['first_last_date_by_source'].items()])
display(coverage)
print('No NOAA-21 data in the audited archive.')

## 18. EDA conclusions and NASA Space Apps relevance

- **Supported starting point:** raw hotspot locations/timestamps, source-separated summaries, and `type=0` filtering.
- **Unsupported interpretations:** one marker equals one distinct fire, FRP equals burned area/total energy, or summed multi-sensor detections equal the nationwide number of fires.
- **Additional evidence needed:** administrative boundary polygons, orbital/cloud-cover information, independent burned-area products, and land-cover time series for the Fire Legacy module.
- **NASA ML scope:** define a grid-plus-time-window target and sensor harmonization baseline; compare statistical and ML models using temporal and geographic holdouts.
- **Human versus natural ignition:** **not supportable** from hotspot CSV alone without verified cause labels, lightning/weather, and land-use evidence.

See `VESTIGIUM_EDA_REPORT_EN.html` or `VESTIGIUM_EDA_COMPLETE_EN.md` for the full 23-section report and all 17 charts.


## 19. Preprocessing contract for frontend and backend

Recommended per-marker output fields: `source`, `satellite`, `version`, `latitude`, `longitude`, `acq_datetime_utc`, `type`, `frp_mw`, `confidence_raw`, `daynight`. At low zoom levels the map should use aggregated clusters; individual points should load by bounded viewport (bbox) and time interval. Monthly timeline data should remain separated by sensor rather than summed as distinct fires.


In [ ]:
# Aggregated data can serve as a preliminary timeline contract; keep sensors separate.
display(monthly.head(10))
assert set(['source','year','month','type0_count','frp_sum_mw']).issubset(monthly.columns)
assert (monthly.type0_count > 0).all()
assert annual.type0_count.sum() == summary['type0_total']
assert daily.type0_count.sum() == summary['type0_total']
print('VALIDATED: type-0 counts agree across daily, monthly, and annual tables.', f"{summary['type0_total']:,}")

## 20. Optional full rerun from the raw ZIP

To re-run the raw audit, place `data.zip` in this directory and uncomment the commands in the next cell. Processing time depends on the computer and does not start automatically when this notebook is opened.

Official attribute references: [NASA MODIS](https://firms.modaps.eosdis.nasa.gov/descriptions/FIRMS_MODIS_Firehotspots.html) and [NASA VIIRS](https://firms.modaps.eosdis.nasa.gov/descriptions/FIRMS_VIIRS_Firehotspots.html).


In [ ]:
# import subprocess, sys
# subprocess.run([sys.executable, 'vestigium_eda.py', 'data.zip', '.'], check=True)
# subprocess.run([sys.executable, 'render_eda.py'], check=True)
print('Raw audit rerun is disabled. Uncomment the two commands above for full reproduction.')